# 05 — The Predictability Horizon

**Can persistent homology forecast where breakthroughs will land?**

We build predictive models using topological features (β₁, persistence entropy)
alongside simple baseline features (cross-class citation count, mixing rate,
patent volume). The key comparison: does topology add information beyond
what simpler metrics provide?

Split: train on pre-2015 breakthroughs, test on post-2015.

---

In [1]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

from src.utils import DATA_DIR, get_logger
from src.breakthroughs import load_breakthroughs
from src.metrics import cpc_mixing_rate, shannon_entropy_cpc
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST

set_style()
logger = get_logger('nb05')

In [2]:
# %% Load data
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')
breakthroughs = load_breakthroughs()

# Ensure citing_year column exists
citations['citing_year'] = pd.to_datetime(citations['citing_date']).dt.year

# Load topology results (re-compute with caching if needed)
from src.topology import compute_all_priority_pairs, PRIORITY_PAIRS
import gc

pair_results = compute_all_priority_pairs(
    citations, cpc_map,
    cache_dir=str(DATA_DIR / 'topology_cache'),
    start_year=1985,
    end_year=2023,
)

# Build topology_results dict keyed by underscore format for breakthrough lookups
topology_results = {}
for pair, group in pair_results.groupby('section_pair'):
    underscore_key = pair.replace('x', '_')
    topology_results[underscore_key] = group.reset_index(drop=True)

print(f'Topology results: {len(topology_results)} CPC pairs')
gc.collect()

2026-03-19 22:27:34 | src.breakthroughs        | INFO    | Loaded 34 breakthroughs from 8 files


Topology results: 10 CPC pairs


0

## Build Feature Matrix

For each CPC section pair × year, compute features and label
whether a breakthrough occurred in that pair within the next 5 years.

In [3]:
# %% Build breakthrough lookup: (section_pair, year) → breakthrough occurred within 5 years
CPC_SECTIONS = list('ABCDEFGH')
bt_lookup = {}

for bt in breakthroughs:
    secs = bt.cpc_sections
    for i in range(len(secs)):
        for j in range(i, len(secs)):
            pair = tuple(sorted([secs[i], secs[j] if j < len(secs) else secs[i]]))
            for yr_offset in range(6):  # 0 to 5 years before filing
                key = (pair, bt.filing_year - yr_offset)
                bt_lookup[key] = 1

In [4]:
# %% Assemble feature matrix
rows = []

for pair_key, topo_df in topology_results.items():
    parts = pair_key.split('_')
    if len(parts) != 2:
        continue
    sec_a, sec_b = parts
    pair = tuple(sorted([sec_a, sec_b]))
    
    # Determine year column
    year_col = 'window_end' if 'window_end' in topo_df.columns else 'year'
    
    for _, row in topo_df.iterrows():
        year = int(row[year_col])
        label = bt_lookup.get((pair, year), 0)
        
        rows.append({
            'section_a': sec_a,
            'section_b': sec_b,
            'year': year,
            # Topological features
            'beta_0': row.get('beta_0', 0),
            'beta_1': row.get('beta_1', 0),
            'beta_2': row.get('beta_2', 0),
            'persistence_entropy': row.get('persistence_entropy', 0),
            'max_persistence_h1': row.get('max_persistence_h1', 0),
            'n_long_lived_h1': row.get('n_long_lived_h1', 0),
            # Simple / structural features
            'n_active_classes': row.get('n_active_classes', 0),
            'mean_distance': row.get('mean_distance', 0),
            'median_distance': row.get('median_distance', 0),
            # Label
            'breakthrough_within_5yr': label,
        })

feat_df = pd.DataFrame(rows)
print(f'Feature matrix: {len(feat_df)} rows')
print(f'Positive labels: {feat_df["breakthrough_within_5yr"].sum()} ({100 * feat_df["breakthrough_within_5yr"].mean():.1f}%)')

Feature matrix: 350 rows
Positive labels: 43 (12.3%)


In [5]:
# %% Evaluation strategy: Stratified K-Fold Cross-Validation
# The temporal split (pre/post 2015) yields only 1 positive test sample — useless.
# Instead, use stratified 5-fold CV on the full dataset for reliable estimates.
from sklearn.model_selection import StratifiedKFold, cross_val_predict

topo_features = ['beta_0', 'beta_1', 'beta_2', 'persistence_entropy', 'max_persistence_h1', 'n_long_lived_h1']
simple_features = ['n_active_classes', 'mean_distance', 'median_distance']
all_features = topo_features + simple_features

X = feat_df[all_features].fillna(0)
y = feat_df['breakthrough_within_5yr']

print(f'Full dataset: {len(X)} rows, {y.sum()} positive ({100*y.mean():.1f}%)')
print(f'Using stratified 5-fold cross-validation')

Full dataset: 350 rows, 43 positive (12.3%)
Using stratified 5-fold cross-validation


## Train Models

Three variants:
1. Topology-only features
2. Simple-only features (patent/citation volume)
3. Combined

In [6]:
# %% Train and evaluate models with stratified 5-fold CV
results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

feature_sets = {
    'Topology only': topo_features,
    'Simple only': simple_features,
    'Combined': all_features,
}

models = {
    'Logistic Regression': lambda: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': lambda: RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
}

for model_name, model_fn in models.items():
    for feat_name, feat_cols in feature_sets.items():
        key = f'{model_name} — {feat_name}'
        
        X_subset = X[feat_cols].values
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_subset)
        
        # Collect out-of-fold predictions
        y_probs = np.zeros(len(y))
        y_preds = np.zeros(len(y), dtype=int)
        
        fold_aucs = []
        for train_idx, test_idx in cv.split(X_scaled, y):
            model = model_fn()
            model.fit(X_scaled[train_idx], y.iloc[train_idx])
            
            if hasattr(model, 'predict_proba'):
                y_probs[test_idx] = model.predict_proba(X_scaled[test_idx])[:, 1]
            else:
                y_probs[test_idx] = model.decision_function(X_scaled[test_idx])
            
            y_preds[test_idx] = model.predict(X_scaled[test_idx])
            
            try:
                fold_aucs.append(roc_auc_score(y.iloc[test_idx], y_probs[test_idx]))
            except ValueError:
                pass
        
        try:
            overall_auc = roc_auc_score(y, y_probs)
        except ValueError:
            overall_auc = 0.5
        
        # Train final model on all data for feature importance
        final_model = model_fn()
        final_model.fit(X_scaled, y)
        
        results[key] = {
            'model': final_model,
            'y_prob': y_probs,
            'y_pred': y_preds,
            'auc': overall_auc,
            'fold_aucs': fold_aucs,
            'features': feat_cols,
        }
        
        fold_str = f' (folds: {", ".join(f"{a:.3f}" for a in fold_aucs)})' if fold_aucs else ''
        print(f'{key}: AUC = {overall_auc:.3f}{fold_str}')

Logistic Regression — Topology only: AUC = 0.738 (folds: 0.760, 0.633, 0.758, 0.865, 0.710)


Logistic Regression — Simple only: AUC = 0.572 (folds: 0.581, 0.619, 0.658, 0.506, 0.488)
Logistic Regression — Combined: AUC = 0.743 (folds: 0.774, 0.734, 0.767, 0.714, 0.725)


Random Forest — Topology only: AUC = 0.915 (folds: 0.916, 0.971, 0.924, 0.948, 0.843)


Random Forest — Simple only: AUC = 0.969 (folds: 0.996, 0.992, 0.957, 0.950, 0.948)


Random Forest — Combined: AUC = 0.960 (folds: 0.976, 0.980, 0.969, 0.968, 0.943)


## Figure 1: ROC Curves

In [7]:
# %% Figure 1: ROC curves (out-of-fold predictions)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, model_name in enumerate(['Logistic Regression', 'Random Forest']):
    ax = axes[idx]
    
    for i, (feat_name, _) in enumerate(feature_sets.items()):
        key = f'{model_name} — {feat_name}'
        if key in results:
            fpr, tpr, _ = roc_curve(y, results[key]['y_prob'])
            ax.plot(fpr, tpr, color=PALETTE_LIST[i], linewidth=2,
                    label=f"{feat_name} (AUC={results[key]['auc']:.3f})")
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(model_name)
    ax.legend(loc='lower right')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

fig.suptitle('Figure 1: ROC Curves — 5-Fold CV, Out-of-Fold Predictions', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '05_roc_curves')

2026-03-19 22:39:48 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/05_roc_curves.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/05_roc_curves.png')

## Figure 2: Feature Importance (SHAP)

In [8]:
# %% Figure 2: Feature importance for the combined Random Forest
rf_combined_key = 'Random Forest — Combined'
if rf_combined_key in results:
    importances = results[rf_combined_key]['model'].feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = [PALETTE['red'] if all_features[i] in topo_features else PALETTE['blue'] for i in sorted_idx]
    ax.barh(range(len(all_features)), importances[sorted_idx], color=colors)
    ax.set_yticks(range(len(all_features)))
    ax.set_yticklabels([all_features[i] for i in sorted_idx])
    ax.set_xlabel('Feature Importance (Gini)')
    ax.set_title('Figure 2: RF Feature Importance (Red = Topology, Blue = Simple)')
    ax.invert_yaxis()
    fig.tight_layout()
    save_figure(fig, '05_feature_importance')

2026-03-19 22:39:48 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/05_feature_importance.png


## Figure 3: Confusion Matrix

In [9]:
# %% Figure 3: Confusion matrix for best model (out-of-fold predictions)
best_key = max(results.keys(), key=lambda k: results[k]['auc'])
best = results[best_key]

cm = confusion_matrix(y, best['y_pred'])

fig, ax = plt.subplots(figsize=(8, 6))
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Breakthrough', 'Breakthrough'],
            yticklabels=['No Breakthrough', 'Breakthrough'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Figure 3: Confusion Matrix — {best_key}\n(5-Fold CV, AUC={best["auc"]:.3f})')
fig.tight_layout()
save_figure(fig, '05_confusion_matrix')

print(f'\nBest model: {best_key}')
print(f'AUC: {best["auc"]:.3f}')
print(f'\n{classification_report(y, best["y_pred"], target_names=["No BT", "BT"])}')

2026-03-19 22:39:49 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/05_confusion_matrix.png



Best model: Random Forest — Simple only
AUC: 0.969

              precision    recall  f1-score   support

       No BT       0.94      0.99      0.97       307
          BT       0.93      0.58      0.71        43

    accuracy                           0.94       350
   macro avg       0.94      0.79      0.84       350
weighted avg       0.94      0.94      0.94       350



## Interpretation

**Key questions this notebook answers:**

1. **Does topology add predictive power?** Compare the AUC of topology-only vs. simple-only vs. combined.
   - If combined >> simple-only: topology adds genuine signal
   - If combined ≈ simple-only: topology is redundant with simpler metrics
   - If all AUC ≈ 0.5: breakthrough prediction is not feasible from citation structure alone

2. **Which features matter most?** SHAP/importance analysis reveals whether β₁,
   persistence entropy, or simpler volume metrics drive predictions.

3. **Honest assessment:**
   - AUC > 0.7 with topology features ranking high → strong finding
   - AUC 0.55-0.65 → weak signal, requires more data or better features
   - AUC ≈ 0.5 → null result, report honestly